In [28]:
import pandas as pd
import re

pd.set_option('display.max_columns', None) # Esto le dice a Pandas que no oculte nada

# 1. Cargar datos
master = pd.read_parquet('../../data/interim/master_merged.parquet')
columnas_iniciales = master.shape[1]

# 2. Filtrar Nulos del Target y Crear Binario
master = master.dropna(subset=['HC70']).copy()
master['TARGET_DESNUTRICION'] = (master['HC70'] < -2.0).astype(int)

# 3. La Guillotina de Nulos (Mantener columnas con al menos 40% de datos válidos)
master = master.dropna(thresh=len(master)*0.40, axis=1)

# 4. Separar Metadatos
metadata_cols = ['UBIGEO', 'LONGITUDX', 'LATITUDY', 'year', 'HHID']
metadata_df = master[metadata_cols].copy()

# 5. Definir variables a ELIMINAR (Leakage, Colinealidad y Redundancia)

# 5.1 Fugas de Datos (Leakage Biológico e Identificadores)
leakage_cols = [
    'HC0', 'HVIDX', 'HV112', 'HV001', 'HV002', 'HV002A', 'NCONGLOME', 'CODCCPP', 'NOMCCPP', 'ID1',
    'HC2', 'HC3', 'HC4', 'HC5', 'HC6', 'HC7', 'HC8', 'HC9', 'HC10', 'HC11', 'HC12', 'HC13', 
    'HC15', 'HC16', 'HC19', 'HC30', 'HC31', 'HC32', 'HC33', 'HC71', 'HC72', 'HC73', 'HC55', 'HC70',
    'HV004', 'HV005', 'hv005', 'HV008', 'HV015', 'HV021', 'HV022', 
    'HV024', 'SHREGION', 'SHPROVIN', 'SHDISTRI', 'SHSEMES', 
    'HV120', 'HV010', 'HV035', 'HC64', 'HV114', 'HC51', 'HC60', 
    'HV218', 'HV117'
]

# 5.2 Alta Colinealidad (Redundantes)
colineales_cols = [
    'HC53', 'HC57',     # Se queda HC56 (Hemoglobina)
    'HC61', 'HV106', 'HV109', # Se quedan HC62 y HV108 (Años de estudio)
    'HV270',            # Se queda HV271 (Score de riqueza continuo)
    'HV009', 'HV013',   # Se queda HV012 (Miembros habituales)
    'HV104', 'HV105',   # Se quedan HC27 (Sexo) y HC1 (Edad en meses)
    'SH71',             # Se queda HV216 (Habitaciones para dormir)
    'QH227A', 'QH227B', # Metadatos de encuestadores
    'HV121', 'HV122', 'HV124', 'HV125', 'HV126', 'HV128' # Variables escolares sobrantes para < 5 años
]

# 5.3 Redundancia de Electrodomésticos y Bienes Menores
redundant_cols = []
for col in master.columns:
    if re.search(r'HV20[6-9]|HV21[0-2]|HV221|HV243A', col, re.IGNORECASE):
        redundant_cols.append(col)
    elif re.search(r'SH51[A-Z]|SH61[A-Z]|SH76[A-Z]', col, re.IGNORECASE):
        redundant_cols.append(col)

# 6. Aplicar la Guillotina Final a TODAS estas variables
todas_a_eliminar = leakage_cols + metadata_cols + colineales_cols + redundant_cols
cols_to_drop = [c for c in todas_a_eliminar if c in master.columns]
master = master.drop(columns=cols_to_drop)

# 7. Imprimir el resultado de la masacre
print("--- REPORTE DE LIMPIEZA INICIAL ---")
print(f"Filas originales: 294,109 -> Filas sin nulos en Target: {len(master)}")
print(f"Columnas iniciales: {columnas_iniciales}")
print(f"Columnas eliminadas: {len(cols_to_drop)}")
print(f"Columnas sobrevivientes para el modelo: {master.shape[1]}")
display(master.head(5))


--- REPORTE DE LIMPIEZA INICIAL ---
Filas originales: 294,109 -> Filas sin nulos en Target: 285284
Columnas iniciales: 183
Columnas eliminadas: 105
Columnas sobrevivientes para el modelo: 56


,HC1,HC27,HC56,HC62,HC63,HV101,HV102,HV103,HV108,HV110,HV111,HV113,HV012,HV014,HV025,HV026,HV040,HV201,HV204,HV205,HV213,HV214,HV215,HV216,HV217,HV219,HV220,HV225,HV226,HV234,HV237,HV237A,HV237B,HV237C,HV237D,HV237E,HV237F,HV237G,HV237X,HV242,HV243C,HV243D,HV244,HV246,HV271,SHTOTH,SH49,SH50,SH63,SH77F,SH227,SH42_Agua_Todo_El_Dia,SH48_Conserva_Agua,SH70_Fuente_Luz,HV240,TARGET_DESNUTRICION
0,20.0,Mujer,NaN,3.0,NaN,Hijo/Hija,Sí,Sí,0.0,No,Sí,Sí,4.0,1.0,Urbano,Pueblo,2335.0,Red dentro de vivienda,0.0,Dentro de la vivienda,Cemento / ladrillo,Adobe o tapia,"Plancha de calamina, fibra de cemento o similares",1.0,más de 3 adultos relacionados,Masculino,26.0,Si,LPG,NaN,Si,Si,No,No,No,No,No,NaN,No,No,No,No,Si,No,0.32939,1.0,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,1
1,28.0,Hombre,10.0,5.0,101.0,Hijo/Hija,Sí,Sí,0.0,No,Sí,Sí,5.0,1.0,Urbano,Pueblo,2335.0,Red dentro de vivienda,0.0,Dentro de la vivienda,Cemento / ladrillo,Adobe o tapia,"Plancha de calamina, fibra de cemento o similares",2.0,más de 3 adultos relacionados,Masculino,38.0,Si,LPG,NaN,Si,Si,No,No,No,No,No,NaN,No,Si,No,No,No,Si,0.41740,1.0,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,1
2,40.0,Mujer,NaN,2.0,NaN,Hijo/Hija,Sí,Sí,0.0,No,Sí,Sí,3.0,1.0,Urbano,Pueblo,2335.0,Red dentro de vivienda,0.0,Dentro de la vivienda,Cemento / ladrillo,Adobe o tapia,"Plancha de calamina, fibra de cemento o similares",2.0,"Dos adultos, dif. Sexo",Masculino,33.0,No,LPG,NaN,Si,Si,No,No,No,No,No,NaN,No,Si,No,No,No,No,0.86784,1.0,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,0
3,52.0,Mujer,9.8,3.0,80.0,Nieto,Sí,Sí,0.0,No,Sí,Sí,5.0,2.0,Rural,Campo,2750.0,Red dentro de vivienda,0.0,Sin servicio (matorral/campo),Tierra / arena,Otro,Tejas,2.0,más de 3 adultos relacionados,Masculino,60.0,NaN,Madera,NaN,No,No,No,No,No,No,No,NaN,No,Si,No,No,Si,Si,-0.97422,1.0,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,2.0,0
4,34.0,Hombre,8.9,3.0,18.0,Nieto,Sí,Sí,0.0,No,Sí,Sí,5.0,2.0,Rural,Campo,2750.0,Red dentro de vivienda,0.0,Sin servicio (matorral/campo),Tierra / arena,Otro,Tejas,2.0,más de 3 adultos relacionados,Masculino,60.0,NaN,Madera,NaN,No,No,No,No,No,No,No,NaN,No,Si,No,No,Si,Si,-0.97422,1.0,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,2.0,1


In [29]:
# 5. Separar Features y Target
features = master.drop(columns=['TARGET_DESNUTRICION'])
target = master['TARGET_DESNUTRICION']

# 6. Preparar tipos de datos para Boosters Modernos
# Ya NO imputamos nulos numéricos, pero para las categóricas CatBoost exige que los NaNs sean texto explícito.
import pandas as pd
for col in features.columns:
    if not pd.api.types.is_numeric_dtype(features[col]):
        # Rellenamos nulos con 'Desconocido' usando fillna, que es 100% infalible en Pandas
        features[col] = features[col].fillna('Desconocido')
        features[col] = features[col].astype(str).astype('category')

print("--- REPORTE DEL PASO 2 ---")
print(f"Total de nulos dejados al natural (solo en numéricas): {features.isnull().sum().sum()}")
print("Tipos de datos actuales:")
print(features.dtypes.value_counts())

print("\nMuestra de tu matriz final:")
display(features.head(5))


--- REPORTE DEL PASO 2 ---
Total de nulos dejados al natural (solo en numéricas): 444914
Tipos de datos actuales:
float64     18
category    15
category     6
category     2
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64

Muestra de tu matriz final:


,HC1,HC27,HC56,HC62,HC63,HV101,HV102,HV103,HV108,HV110,HV111,HV113,HV012,HV014,HV025,HV026,HV040,HV201,HV204,HV205,HV213,HV214,HV215,HV216,HV217,HV219,HV220,HV225,HV226,HV234,HV237,HV237A,HV237B,HV237C,HV237D,HV237E,HV237F,HV237G,HV237X,HV242,HV243C,HV243D,HV244,HV246,HV271,SHTOTH,SH49,SH50,SH63,SH77F,SH227,SH42_Agua_Todo_El_Dia,SH48_Conserva_Agua,SH70_Fuente_Luz,HV240
0,20.0,Mujer,NaN,3.0,NaN,Hijo/Hija,Sí,Sí,0.0,No,Sí,Sí,4.0,1.0,Urbano,Pueblo,2335.0,Red dentro de vivienda,0.0,Dentro de la vivienda,Cemento / ladrillo,Adobe o tapia,"Plancha de calamina, fibra de cemento o similares",1.0,más de 3 adultos relacionados,Masculino,26.0,Si,LPG,NaN,Si,Si,No,No,No,No,No,Desconocido,No,No,No,No,Si,No,0.32939,1.0,NaN,NaN,Desconocido,No,Desconocido,Desconocido,Desconocido,NaN,NaN
1,28.0,Hombre,10.0,5.0,101.0,Hijo/Hija,Sí,Sí,0.0,No,Sí,Sí,5.0,1.0,Urbano,Pueblo,2335.0,Red dentro de vivienda,0.0,Dentro de la vivienda,Cemento / ladrillo,Adobe o tapia,"Plancha de calamina, fibra de cemento o similares",2.0,más de 3 adultos relacionados,Masculino,38.0,Si,LPG,NaN,Si,Si,No,No,No,No,No,Desconocido,No,Si,No,No,No,Si,0.41740,1.0,NaN,NaN,Desconocido,No,Desconocido,Desconocido,Desconocido,NaN,NaN
2,40.0,Mujer,NaN,2.0,NaN,Hijo/Hija,Sí,Sí,0.0,No,Sí,Sí,3.0,1.0,Urbano,Pueblo,2335.0,Red dentro de vivienda,0.0,Dentro de la vivienda,Cemento / ladrillo,Adobe o tapia,"Plancha de calamina, fibra de cemento o similares",2.0,"Dos adultos, dif. Sexo",Masculino,33.0,No,LPG,NaN,Si,Si,No,No,No,No,No,Desconocido,No,Si,No,No,No,No,0.86784,1.0,NaN,NaN,Desconocido,No,Desconocido,Desconocido,Desconocido,NaN,NaN
3,52.0,Mujer,9.8,3.0,80.0,Nieto,Sí,Sí,0.0,No,Sí,Sí,5.0,2.0,Rural,Campo,2750.0,Red dentro de vivienda,0.0,Sin servicio (matorral/campo),Tierra / arena,Otro,Tejas,2.0,más de 3 adultos relacionados,Masculino,60.0,Desconocido,Madera,NaN,No,No,No,No,No,No,No,Desconocido,No,Si,No,No,Si,Si,-0.97422,1.0,NaN,NaN,Desconocido,No,Desconocido,Desconocido,Desconocido,NaN,2.0
4,34.0,Hombre,8.9,3.0,18.0,Nieto,Sí,Sí,0.0,No,Sí,Sí,5.0,2.0,Rural,Campo,2750.0,Red dentro de vivienda,0.0,Sin servicio (matorral/campo),Tierra / arena,Otro,Tejas,2.0,más de 3 adultos relacionados,Masculino,60.0,Desconocido,Madera,NaN,No,No,No,No,No,No,No,Desconocido,No,Si,No,No,Si,Si,-0.97422,1.0,NaN,NaN,Desconocido,No,Desconocido,Desconocido,Desconocido,NaN,2.0


In [30]:
import sys
import os
import re
import unicodedata

# 1. Le decimos a Python dónde está tu carpeta 'mnp'
sys.path.append(os.path.abspath('../../')) 
from mnp.configs.column_labels import LABELS

# 2. Renombramos todas las columnas matemáticas a Español
features = features.rename(columns=LABELS)

# 3. Limpiar y aplanar los nombres (sin tildes, con guiones bajos)
clean_cols = []
for col in features.columns:
    # Quitar tildes y acentos
    col_str = unicodedata.normalize('NFKD', str(col)).encode('ASCII', 'ignore').decode('utf-8')
    # Reemplazar todo lo que no sea letra o número por guion bajo
    col_str = re.sub(r'[^a-zA-Z0-9]+', '_', col_str)
    # Eliminar guiones bajos al inicio y al final
    col_str = col_str.strip('_')
    clean_cols.append(col_str)

features.columns = clean_cols

print("¡Columnas renombradas y aplanadas exitosamente!")
display(features.head(2))


¡Columnas renombradas y aplanadas exitosamente!


,HC1_Edad_en_meses,HC27_Sexo,HC56_Nivel_de_hemoglobina_ajustado_por_altitud,HC62_Ano_mas_alto_de_educacion_de_la_madre,HC63_Intervalo_de_nacimientos_anteriores_al_nino,HV101_Parentesco_con_jefe_de_hogar,HV102_Residente_habitual,HV103_Durmio_aqui_anoche,HV108_Numero_de_anos_de_estudio,HV110_Asiste_a_escuela,HV111_Esta_viva_la_madre_natural,HV113_Esta_vivo_el_padre_natural,HV012_Miembros_habituales_De_jure,HV014_Ninos_menores_de_5_anos,HV025_Area_de_residencia,HV026_Lugar_de_residencia,HV040_Altitud_del_conglomerado_en_metros,HV201_Fuente_principal,HV204_Tiempo_de_viaje_a_fuente,HV205_Tipo_de_servicio_higienico,HV213_Material_predominante_en_el_piso,HV214_Material_predominante_en_la_pared,HV215_Material_predominante_en_el_techo,HV216_Habitaciones_para_dormir,HV217_Estructura_de_relacion,HV219_Sexo_del_jefe_del_hogar,HV220_Edad_del_jefe_del_hogar,HV225_Comparte_servicio_higienico,HV226_Tipo_de_combustible_para_cocinar,HV234_Prueba_de_yodo_para_sal,HV237_Tratamiento_del_agua,HV237A_Tratamiento_de_agua_hervir,HV237B_Anadir_lejia_o_cloro,HV237C_Filtrar_por_pano,HV237D_Usar_filtro_de_agua,HV237E_Desinfeccion_solar,HV237F_Dejar_reposar,HV237G_Agua_embotellada,HV237X_Tratamiento_de_agua_otros,HV242_Cuarto_separado_para_cocinar,HV243C_Tiene_carreta_jalada_por_animales,HV243D_Tiene_bote_a_motor,HV244_Dueno_de_tierras_agricolas,HV246_Dueno_de_ganado_animales,HV271_Factor_de_puntuacion_del_indice_de_riqueza,SHTOTH_Hogares_en_la_vivienda,SH49_Tipo_de_envase_o_recipiente,SH50_Lo_usa_con_tapa,SH63_Utiliza_otro_tipo_de_combustible_para_cocinar,SH77F_Otro_tipo_de_transporte_caballos_peque_peque_etc,SH227_Prueba_de_cloro,SH42_Agua_Todo_El_Dia,SH48_Conserva_Agua,SH70_Fuente_Luz,HV240_Tiene_chimenea_o_campana
0,20.0,Mujer,NaN,3.0,NaN,Hijo/Hija,Sí,Sí,0.0,No,Sí,Sí,4.0,1.0,Urbano,Pueblo,2335.0,Red dentro de vivienda,0.0,Dentro de la vivienda,Cemento / ladrillo,Adobe o tapia,"Plancha de calamina, fibra de cemento o similares",1.0,más de 3 adultos relacionados,Masculino,26.0,Si,LPG,NaN,Si,Si,No,No,No,No,No,Desconocido,No,No,No,No,Si,No,0.32939,1.0,NaN,NaN,Desconocido,No,Desconocido,Desconocido,Desconocido,NaN,NaN
1,28.0,Hombre,10.0,5.0,101.0,Hijo/Hija,Sí,Sí,0.0,No,Sí,Sí,5.0,1.0,Urbano,Pueblo,2335.0,Red dentro de vivienda,0.0,Dentro de la vivienda,Cemento / ladrillo,Adobe o tapia,"Plancha de calamina, fibra de cemento o similares",2.0,más de 3 adultos relacionados,Masculino,38.0,Si,LPG,NaN,Si,Si,No,No,No,No,No,Desconocido,No,Si,No,No,No,Si,0.41740,1.0,NaN,NaN,Desconocido,No,Desconocido,Desconocido,Desconocido,NaN,NaN


In [31]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import time

# Identificar las columnas categóricas para CatBoost
cat_features = features.select_dtypes(include=['category']).columns.tolist()

# Split temporal (Train <= 2018, Test > 2018)
train_mask = metadata_df['year'] <= 2018
test_mask = metadata_df['year'] > 2018

X_train = features[train_mask]
y_train = target[train_mask]
X_test = features[test_mask]
y_test = target[test_mask]

print(f"Train (<=2018): {X_train.shape}")
print(f"Test (>2018): {X_test.shape}")

models = {
    # XGBoost: requiere enable_categorical=True
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss', enable_categorical=True),
    # LightGBM: lo detecta en automático si son 'category'
    'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0)
}

for name, model in models.items():
    print(f"Entrenando {name}...")
    start_time = time.time()
    
    # CatBoost necesita que le pasemos explícitamente cuáles son las categóricas
    if name == 'CatBoost':
        model.fit(X_train, y_train, cat_features=cat_features)
    else:
        model.fit(X_train, y_train)
        
    print(f"[{name}] Completado en {time.time() - start_time:.2f} segundos.\n")


Train (<=2018): (165234, 55)
Test (>2018): (120050, 55)
Entrenando XGBoost...
[XGBoost] Completado en 2.04 segundos.

Entrenando LightGBM...
[LightGBM] Completado en 0.86 segundos.

Entrenando CatBoost...
[CatBoost] Completado en 219.87 segundos.



In [32]:
from sklearn.metrics import classification_report, roc_auc_score

print("--- MÉTRICAS DE EVALUACIÓN EN DATOS INVISIBLES (Años > 2018) ---")

for name, model in models.items():
    print(f"\n==================== {name} ====================")
    
    # El modelo intenta predecir la desnutrición de los niños nuevos
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] # Probabilidad matemática
    
    # Métrica AUC-ROC
    auc = roc_auc_score(y_test, y_proba)
    print(f"➜ AUC-ROC Score: {auc:.4f}")
    
    # Reporte Clínico Detallado
    print(classification_report(y_test, y_pred, target_names=['Sanos (0)', 'Desnutridos (1)']))


--- MÉTRICAS DE EVALUACIÓN EN DATOS INVISIBLES (Años > 2018) ---

==================== XGBoost ====================
➜ AUC-ROC Score: 0.7478
                 precision    recall  f1-score   support

      Sanos (0)       0.88      0.99      0.93    104856
Desnutridos (1)       0.50      0.08      0.14     15194

       accuracy                           0.87    120050
      macro avg       0.69      0.53      0.53    120050
   weighted avg       0.83      0.87      0.83    120050


==================== LightGBM ====================
➜ AUC-ROC Score: 0.7561
                 precision    recall  f1-score   support

      Sanos (0)       0.88      0.99      0.93    104856
Desnutridos (1)       0.54      0.07      0.12     15194

       accuracy                           0.87    120050
      macro avg       0.71      0.53      0.53    120050
   weighted avg       0.84      0.87      0.83    120050


==================== CatBoost ====================
➜ AUC-ROC Score: 0.7576
                 p